# Transformer als Funktion – von der Wahrscheinlichkeit zum Wort

**Fach:** KIT (Künstliche Intelligenz, Informatik und Technologie)

**Lernziele dieser Stunde:**
- Ihr versteht, dass ein Sprachmodell einem Satzanfang keine feste Antwort, sondern eine **Wahrscheinlichkeitsverteilung** über mögliche nächste Wörter zuordnet.
- Ihr könnt **Greedy Decoding** und **Sampling** unterscheiden und erklären, warum nur eines von beiden deterministisch ist.
- Ihr könnt das Verhalten an einem echten, kleinen Sprachmodell experimentell nachvollziehen.

## 1. Hinführung

**Impuls – Diskutiert zu zweit (2 Minuten):**

> Wenn man ChatGPT zweimal exakt dieselbe Frage stellt, bekommt man manchmal zwei unterschiedliche Antworten. Stellt man dieselbe Frage an ein einfaches Taschenrechner-Programm, kommt aber immer dasselbe Ergebnis heraus.
>
> **Fragen:**
> 1. Ist ein Sprachmodell wie ChatGPT damit "kaputt" oder "unzuverlässig"?
> 2. Was müsste ein Sprachmodell intern eigentlich "tun", damit unterschiedliche Antworten bei gleicher Frage möglich sind?

Haltet eure Vermutungen kurz schriftlich fest – wir vergleichen sie am Ende der Stunde mit dem, was ihr dann wisst.

*Notizen / erste Vermutung:*

```


```

## 2. Theorieteil

### 2.1 Der Transformer als Funktion

Ein Sprachmodell (Transformer) wird trainiert, um für einen gegebenen Satzanfang **jedem** Wort im Vokabular (typischerweise mehrere Zehntausend Wörter/Wortteile) eine Wahrscheinlichkeit zuzuordnen: *Wie wahrscheinlich ist dieses Wort als nächstes?*

Das Ergebnis ist also **keine einzelne Antwort**, sondern eine vollständige Wahrscheinlichkeitsverteilung über das gesamte Vokabular – vergleichbar mit einem sehr großen, unfairen Würfel, bei dem jede Seite (= jedes mögliche Wort) eine eigene, vom Modell berechnete Wahrscheinlichkeit hat.

**Wichtig für den Funktionsbegriff:**

$$\text{Transformer}(\text{Satzanfang}) = \text{Wahrscheinlichkeitsverteilung über alle möglichen nächsten Wörter}$$

Diese Berechnung ist tatsächlich eine echte (deterministische) Funktion: Bei gleichem Input kommt immer dieselbe Verteilung heraus.

### 2.2 Vom Wörterbuch zum Wort – die Decoding-Strategie

Aus dieser Verteilung muss anschließend **ein konkretes** Wort ausgewählt werden. Das übernimmt ein zweiter, separater Mechanismus – die sogenannte *Decoding-Strategie*. Die zwei wichtigsten:

| Strategie | Wie wird gewählt? | Deterministisch? |
|---|---|---|
| **Greedy Decoding** | Es wird immer das Wort mit der höchsten Wahrscheinlichkeit gewählt (Maximum). | Ja – gleicher Input → immer gleicher Output |
| **Sampling** | Es wird gemäß der Verteilung "gewürfelt" (Wörter mit höherer Wahrscheinlichkeit werden häufiger gezogen, aber nicht garantiert). | Nein – gleicher Input kann zu unterschiedlichen Outputs führen |

Ein zusätzlicher Parameter beim Sampling ist die **Temperatur**: Sie steuert, wie "scharf" (temperatur < 1, bevorzugt wahrscheinliche Wörter stark) oder wie "gleichmäßig" (temperatur > 1, gibt auch unwahrscheinlichen Wörtern mehr Chancen) gewürfelt wird.

### 2.3 Zusammenfassend

```
Satzanfang  --[Transformer, deterministisch]-->  Wahrscheinlichkeitsverteilung
Wahrscheinlichkeitsverteilung --[Decoding-Strategie]--> ein konkretes Wort
```

Ob am Ende bei gleicher Eingabe immer dasselbe Wort herauskommt, hängt also **nicht** vom Transformer selbst ab, sondern davon, welche Decoding-Strategie danach verwendet wird.

### 2.4 Schematische Darstellung

Führt die folgende Zelle aus, um die Pipeline als Schaubild zu sehen (keine Eingabe nötig).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 2)
ax.axis("off")

boxen = [
    (0.2, "Satzanfang", "#DDEBF7"),
    (2.6, "Transformer\n(deterministisch)", "#4C72B0"),
    (5.4, "Wahrscheinlichkeits-\nverteilung", "#DDEBF7"),
    (8.0, "Decoding-Strategie\n(Greedy / Sampling)", "#C44E52"),
]

breite, hoehe = 2.1, 1.2
for x, text, farbe in boxen:
    schriftfarbe = "white" if farbe in ("#4C72B0", "#C44E52") else "black"
    box = mpatches.FancyBboxPatch((x, 0.4), breite, hoehe,
                                   boxstyle="round,pad=0.05",
                                   facecolor=farbe, edgecolor="black")
    ax.add_patch(box)
    ax.text(x + breite / 2, 0.4 + hoehe / 2, text, ha="center", va="center",
            fontsize=9, color=schriftfarbe)

pfeil_y = 1.0
for x_start in [2.3, 4.9, 7.7]:
    ax.annotate("", xy=(x_start + 0.3, pfeil_y), xytext=(x_start, pfeil_y),
                 arrowprops=dict(arrowstyle="->", lw=2))

plt.title("Vom Satzanfang zum ausgegebenen Wort", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Ausprobieren

Wir laden jetzt ein kleines, echtes Sprachmodell und beobachten das eben erklärte Verhalten live.

In [ ]:
# Falls noch nicht installiert (einmalig ausfuehren, dann Kommentar wieder setzen)
# !pip install transformers torch matplotlib --quiet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Kleines, CPU-taugliches deutsches Modell
MODEL_NAME = "dbmdz/german-gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()  # Trainingsmodus aus, nur Inferenz

print(f"Modell geladen: {MODEL_NAME}")
print(f"Vokabulargroesse: {model.config.vocab_size} moegliche Tokens")

### 3.1 Die Verteilung berechnen und anzeigen

Die folgende Funktion zeigt genau das, was der Transformer *tatsächlich* berechnet.

In [ ]:
def naechste_token_verteilung(text, top_k=10):
    """
    Gibt die top_k wahrscheinlichsten naechsten Tokens fuer einen Satzanfang zurueck,
    zusammen mit ihren Wahrscheinlichkeiten.
    """
    eingabe = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        ausgabe = model(**eingabe)
        logits = ausgabe.logits[0, -1, :]                     # Logits fuer das naechste Token
        wahrscheinlichkeiten = torch.softmax(logits, dim=-1)   # -> echte Verteilung (Summe = 1)

    top_werte, top_indizes = torch.topk(wahrscheinlichkeiten, top_k)

    ergebnisse = []
    for wert, index in zip(top_werte, top_indizes):
        token_text = tokenizer.decode([index])
        ergebnisse.append((token_text, wert.item()))

    return ergebnisse


def zeige_verteilung(text, top_k=10):
    ergebnisse = naechste_token_verteilung(text, top_k=top_k)
    tokens = [t for t, _ in ergebnisse]
    werte = [p for _, p in ergebnisse]

    plt.figure(figsize=(9, 4))
    balken = plt.bar(tokens, werte, color="#4C72B0")
    balken[0].set_color("#C44E52")  # wahrscheinlichstes Token hervorheben

    plt.title(f"Wahrscheinlichkeitsverteilung fuer: \u201e{text} \u2026\u201c")
    plt.ylabel("Wahrscheinlichkeit")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


satzanfang = "Der Hund läuft über die"
zeige_verteilung(satzanfang, top_k=10)

**Beobachtung:** Notiert kurz, welches Wort am wahrscheinlichsten ist und ob euch das Ergebnis plausibel erscheint.

### 3.2 Greedy Decoding – deterministisch

In [ ]:
def greedy_naechstes_token(text):
    top1 = naechste_token_verteilung(text, top_k=1)
    return top1[0][0]


print("Greedy Decoding – 5x derselbe Satzanfang:")
for i in range(5):
    token = greedy_naechstes_token(satzanfang)
    print(f"  Durchlauf {i+1}: {token!r}")

### 3.3 Sampling – nicht-deterministisch

In [ ]:
def sample_naechstes_token(text, temperatur=1.0, top_k=10):
    ergebnisse = naechste_token_verteilung(text, top_k=top_k)
    tokens = [t for t, _ in ergebnisse]
    werte = torch.tensor([p for _, p in ergebnisse])

    werte = werte ** (1.0 / temperatur)  # Temperatur wirkt auf die Verteilung
    werte = werte / werte.sum()

    index = torch.multinomial(werte, num_samples=1).item()
    return tokens[index]


print("Sampling – 5x derselbe Satzanfang:")
for i in range(5):
    token = sample_naechstes_token(satzanfang, temperatur=1.0)
    print(f"  Durchlauf {i+1}: {token!r}")

**Beobachtung:** Vergleicht die Ausgaben aus 3.2 und 3.3. Bestätigt sich, was in Kapitel 2 erklärt wurde?

## 4. Aufgaben

### Pflichtaufgaben

**Aufgabe 1:** Wählt einen eigenen Satzanfang (z.B. aus eurem Lieblingsthema) und lasst euch mit `zeige_verteilung(...)` die Top-10-Verteilung anzeigen. Beschreibt in 1–2 Sätzen, was ihr seht.

**Aufgabe 2:** Führt für euren Satz aus Aufgabe 1 fünfmal `greedy_naechstes_token(...)` aus. Was stellt ihr fest? Begründet mit dem Fachbegriff aus Kapitel 2.

**Aufgabe 3:** Führt für denselben Satz fünfmal `sample_naechstes_token(..., temperatur=1.0)` aus. Vergleicht das Ergebnis mit Aufgabe 2 und erklärt den Unterschied.

### Zusatzaufgaben

**Aufgabe 4:** Verändert die Temperatur beim Sampling (z.B. `temperatur=0.2` und `temperatur=2.0`) für denselben Satz. Beschreibt, wie sich das Verhalten der Ausgabe verändert und warum (Tipp: schaut euch nochmal die Formel `werte ** (1.0 / temperatur)` an).


### Transferaufgabe / Reflexion

**Aufgabe 5:** Erklärt in eigenen, vollständigen Sätzen, warum die Aussage "ChatGPT ist nicht deterministisch, also kann der Transformer dahinter auch keine Funktion sein" **falsch** ist. Bezieht euch dabei auf die Unterscheidung zwischen Transformer-Berechnung und Decoding-Strategie.

**Aufgabe 6:** Vergleicht eure ursprüngliche Vermutung aus Kapitel 1 mit eurem heutigen Wissensstand. Was habt ihr dazugelernt, was war schon richtig?

---
## Hinweise für die Lehrkraft (nicht für Schüler gedacht)

- **Zeitbedarf:** Hinführung ca. 5 Min, Theorie ca. 15 Min (Frontal/Lehrgespräch), Ausprobieren ca. 15 Min (Partnerarbeit am Rechner), Aufgaben ca. 20–30 Min je nach Kursniveau.
- **Erwartete Antwort Aufgabe 5:** Der Transformer berechnet deterministisch eine Verteilung; die Zufälligkeit entsteht ausschließlich im nachgelagerten Sampling-Schritt, nicht im neuronalen Netz selbst.
- **Differenzierung**:für leistungsschwächere Gruppen kann Aufgabe 4 auf eine feste Temperatur-Vorgabe reduziert werden.